# LLM に生成文の品質を採点させる (LLM-as-a-judge)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/18_evaluation/llm_judge.ipynb)

NLP クックブック 18-12 ／ 評価指標と統計的検定

> API キーが必要 (Colab 左の 🔑 シークレットに登録)

要約や翻訳、対話応答の品質を、参照訳との一致ではなく、LLM に評価基準を与えて採点させたい。

In [ ]:
!pip install -q google-genai pydantic scipy

In [ ]:
from typing import Literal
from google import genai
from google.colab import userdata
from google.genai import types
from pydantic import BaseModel

# Google AI Studio で作った API キーを、Colab のシークレットに GOOGLE_API_KEY として登録しておく (無料枠で使える)
client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))
MODEL = "gemini-flash-latest"

class Judgment(BaseModel):
    reasoning: str                    # 先に根拠を書かせてから点数を付けさせる
    faithfulness: Literal[1, 2, 3, 4, 5]
    fluency: Literal[1, 2, 3, 4, 5]

RUBRIC = """あなたは要約の評価者です。元の文章と要約を読み、次の 2 つの観点を 1〜5 の整数で採点してください。
- faithfulness (忠実性): 要約の内容が元の文章と矛盾せず、書かれていない情報を含まないか。5 = 完全に忠実、1 = 重大な誤りがある。
- fluency (流暢性): 日本語として自然で読みやすいか。5 = 完全に自然、1 = 意味が取れない。
要約が短いことや情報が少ないことは減点しないでください。"""

def judge(source, summary):
    r = client.models.generate_content(
        model=MODEL, contents=f"# 元の文章\n{source}\n\n# 要約\n{summary}",
        config=types.GenerateContentConfig(system_instruction=RUBRIC, temperature=0.0,
                                           response_mime_type="application/json", response_schema=Judgment),
    )
    return r.parsed

source = "同社の 2023 年度の売上高は前年比 5% 増の 120 億円だった。一方、原材料費の高騰により営業利益は 8 億円と、前年の 10 億円から減少した。"
for summary in ["売上高は 5% 増の 120 億円だったが、営業利益は 8 億円に減少した。", "売上高、営業利益ともに前年を上回り、好調だった。",
                "売上高 120 億円は 5% 増で営業利益が 8 億円の減少を原材料費だった。"]:
    j = judge(source, summary)
    print(f"忠実性 {j.faithfulness} / 流暢性 {j.fluency} | {summary}\n   {j.reasoning[:80]}")

In [ ]:
# 評価器そのものを検証する: 人手の評価との相関を必ず確認する
from scipy.stats import spearmanr

human = [5, 1, 3]                      # 人間が付けた忠実性スコア (実際には 100 件以上で確認する)
llm = [5, 1, 4]                        # judge() の faithfulness
print("Spearman:", round(spearmanr(human, llm).statistic, 3))

- 採点基準 (ルーブリック) を具体的に書く。観点ごとに分けて採点させ、点数の前に根拠を書かせると安定する。
- LLM の評価には偏りがある (長い出力を好む、自分と同系統のモデルの出力に甘い、提示順に影響される)。2 つの出力を比べさせる場合は、順序を入れ替えて 2 回評価する。
- 人手評価との一致を確認していない LLM 評価は、評価として使ってはいけない。
- 評価されるモデルと評価するモデルは、できれば別系統のものにする。同じ書き方で、Claude API など他の API のモデルにも置き換えられる (LLM の API の章の「Claude API を呼び出す」を参照)。Gemini API の使い方と構造化出力は、LLM の API の章の「Gemini API を呼び出す (無料枠)」と「Gemini に決まった形式 (JSON スキーマ) で出力させる」を参照。
- 無料枠で送った内容は Google のモデルの改善に使われることがある。評価するデータに個人情報や機密が含まれる場合は送らない。